# Ingestion PDF

## Loader sait : 

- où trouver le fichier 
- comment l'ouvrir 
- comment lire sas structure interne
- et comment en extraire le texte

## PyPDFLoader

In [ ]:
from langchain_community.document_loaders import PyPDFLoader 

In [ ]:
pdf_path = "../data/pdf_files/rapport-activite_credit_mutuelle.pdf"

In [ ]:
loader = PyPDFLoader(pdf_path)
pdf_docs = loader.load()

In [ ]:
print("Nombre de pages :", len(pdf_docs))
print(pdf_docs[3].page_content[:500])
print(pdf_docs[0].metadata)

## Ingestion avancée avec PyMUPDFLoader

In [ ]:
from langchain_community.document_loaders import PyMuPDFLoader

pdf_path = "../data/pdf_files/rapport-activite_credit_mutuelle.pdf"
loader2 = PyMuPDFLoader(pdf_path)
pdfmu_docs = loader2.load()

print("Nombre de pages :", len(pdfmu_docs))
#print(pdfmu_docs[0].page_content[:500])
#print(pdfmu_docs[3].page_content[:500])
print(pdfmu_docs[3].metadata)

## Fonction de nettoyage

In [ ]:
import re
import unicodedata

def clean_pdf_text(text: str) -> str:
    # 0. Normaliser les caractères unicode (très important pour les points PDF)
    text = unicodedata.normalize("NFKC", text)

    # 1. Supprimer tous les points et assimilés (Unicode category: Po = punctuation other)
    text = re.sub(r"[.\u2022\u2027\u2219\u00B7\u2043\u25CF\u25E6\u2024\uFE52\uFF0E]+", " ", text)

    # 2. Supprimer les traits d’union PDF
    text = re.sub(r"-\s+", "", text)

    # 3. Supprimer les multiples espaces
    text = re.sub(r"\s+", " ", text)

    # 4. Supprimer les séparateurs horizontaux
    text = re.sub(r"[_=]{3,}", " ", text)

    # 5. Supprimer les numéros de page
    text = re.sub(r"Page\s*\d+\s*/\s*\d+", " ", text, flags=re.IGNORECASE)

    # 6. Nettoyage final
    text = text.strip()
    return text


## Appliquer le nettoyage 


In [ ]:
cleaned_docs = []

for doc in pdf_docs:
    cleaned = clean_pdf_text(doc.page_content)
    doc.page_content = cleaned
    cleaned_docs.append(doc)

## Avant/Après

In [ ]:
print("AVANT :")
print(pdfmu_docs[11].page_content[:300])

print("\nAPRÈS :")
print(cleaned_docs[11].page_content[:300])

## CHUNKING

LES VALEURS OPTIMALES :

- chunk_size = entre 1000 et 1200 caractères

- overlap = entre 100 et 200 caractères

Par exemple :

chunk 1 : phrase A + début de phrase B

chunk 2 : fin de phrase B + phrase C

## IMPLEMENTATION DU CHUNCKING

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

In [ ]:
splitter = RecursiveCharacterTextSplitter(
    chunk_size = 1200,
    chunk_overlap = 150,
    separators = ["\n\n", "\n", ".", " ", ""]
)

In [ ]:
chunks = splitter.split_documents(cleaned_docs)

print("Nombre de chunks :", len(chunks))
print(chunks[15].page_content)

In [ ]:
for i, c in enumerate(chunks[:5]): 
    print(f"---- CHUNK{i} -----")
    print(c.page_content[:400])
    print()